# 11 — Evidence, подпись и WORM-аудит — Spillety (Elliptic++)

**Контекст Spillety:** 203 769 транзакций `elliptic_txs_*`, `time_step` 1..49, 234 356 рёбер. Цель ноутбука — показать полный путь от алерта до проверяемого доказательства: Evidence JSON → криптоподпись → Merkle-дерево → WORM-лог с якорением. Всё на детерминированных `hashlib`/`hmac` без внешних крипто-зависимостей.

**Зачем:** регулятору (SAR) нужен не скор, а *проверяемое* доказательство: что риск посчитан именно этой моделью, когда и на каких данных, с якорем и причинным путём. Без этого алерт — мнение; с подписью+Merkle+OTS — артефакт для аудита.

**План:**
- §1 Загрузка через `load_elliptic` → `temporal_split` 1..30/31..40/41..49 → RandomForest(100) как GBDT-proxy → 100 алертов (`score>0.5`) → Evidence JSON.
- §2 Пример Evidence + валидация схемы (jsonschema-proxy).
- §3 Подпись `HMAC-SHA256` как proxy `Ed25519` (64 байта mock) + tamper-демо.
- §4 Merkle-дерево из 100 хешей → root → proof для #0 (`⌈log₂100⌉=7`) + график `proof size vs N`.
- §5 WORM proxy (append-only log + `worm_root.txt` как proxy OpenTimestamps→Bitcoin) + audit trail.
- §6 Метрики: время верификации, размер proof, доля верифицированных + выводы (ponytail Ed25519+Merkle+OTS).

> **Крипто-proxy:** вместо `Ed25519` (требует `cryptography`/`PyNaCl`) используем `HMAC-SHA256` — детерминированный, без зависимостей, с тем же контрактом `sign(key, hash) → sig` / `verify(key, hash, sig)`. 32-байтовый HMAC расширяем до 64 байт mock конкатенацией `HMAC(key, h) || HMAC(key, HMAC)` — размер как у Ed25519. Замена drop-in: поменяй `hmac` на `ed25519` без смены интерфейса.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = lambda x: print(x)
from pathlib import Path
import base64
import hashlib
import hmac
import json
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT (работает из docs/notebooks и из корня)
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Elliptic data not found, tried: {candidates}")
print(f"DATA_ROOT = {DATA_ROOT.resolve()}")

# крипто-proxy параметры (демо-ключ, в проде — HSM / Ed25519 privkey)
DEMO_KEY = b"spillety-demo-key-32-bytes!!1234"  # 32 байта
print(f"DEMO_KEY len={len(DEMO_KEY)}  (HMAC-SHA256 proxy Ed25519, mock 64B sig)")
print(f"hashlib: {hashlib.sha256(b'test').hexdigest()[:16]}...  hmac ok={hmac.new(DEMO_KEY, b'test', hashlib.sha256).hexdigest()[:16]}...")


## 1. Загрузка через loader, обучение RF и генерация 100 Evidence JSON

Читаем только через `load_elliptic` (immutable source). Фильтр `class ∈ {1,2}`, `y=1 iff illicit`. Temporal split фиксирован 1..30 train / 31..40 valid / 41..49 test — без шаффла. На train обучаем `RandomForest(n_estimators=100, random_state=72)` как proxy LightGBM/GBDT (без `lightgbm` зависимости, тот же табличный лес, `predict_proba`). На test берём 100 алертов с `score>0.5` (top-100 по убыванию; если <100 — берём top-100).

Для каждого алерта формируем Evidence JSON proxy:
- `alert_id, risk_score, tier` — скор и пороговый тир (`high>0.8 / medium>0.5`).
- `anchors: {distance, source}` — ближайший граф-сосед (edgelist) + Euclidean distance в 165-мерном пространстве признаков.
- `causal_path: {edge, effect}` — ребро `tx->anchor` и эффект (proxy причинного вклада).
- `shap_proxy: {feature importances}` — топ-5 `feature_importances_` из RF, взвешенных на скор.
- `provenance: {model_version, date, source, train_range}` — откуда, когда и чем посчитано.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features {features.shape}  classes {classes.shape}  edgelist {edgelist.shape}  merged {merged.shape}")
print(f"time {merged['time_step'].min()}..{merged['time_step'].max()}  classes: {merged['class'].astype(str).value_counts().to_dict()}")

df = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df["y"] = (df["class"].astype(str) == "1").astype(int)
feat_cols = [c for c in df.columns if c.startswith("feat_")]
print(f"labeled {len(df):,}  illicit {df['y'].mean():.2%}  feat {len(feat_cols)}")

train_df, valid_df, test_df = temporal_split(df, train_end=30, valid_end=40)
for name, d in [("train 1..30", train_df), ("valid 31..40", valid_df), ("test  41..49", test_df)]:
    print(f"{name}: n={len(d):,}  illicit={d['y'].mean():.4f} ({d['y'].sum():,})")

# граф-соседи для anchors
from collections import defaultdict
nbr_map = defaultdict(list)
for a, b in edgelist[["txId1", "txId2"]].values:
    nbr_map[int(a)].append(int(b))
    nbr_map[int(b)].append(int(a))  # undirected для поиска якоря

feat_by_id = dict(zip(features["txId"].astype(int), features[[c for c in features.columns if c.startswith("feat_")]].values))
# также маппим score якоря если он labeled (иначе NaN)
test_scores_placeholder = {}  # заполним после обучения

# обучение RF (GBDT proxy)
X_train = train_df[feat_cols].values
y_train = train_df["y"].values
X_test = test_df[feat_cols].values
y_test = test_df["y"].values
test_tx = test_df["txId"].astype(int).values
test_times = test_df["time_step"].values

rf = RandomForestClassifier(n_estimators=100, random_state=72, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"\nRF fitted: {rf.n_estimators} trees, depth≈{rf.estimators_[0].get_depth()}")

proba_test = rf.predict_proba(X_test)[:, 1]
# 100 алертов score>0.5, top-100 по скору
mask_alert = proba_test > 0.5
idx_alert = np.where(mask_alert)[0]
if len(idx_alert) < 100:
    # fallback — топ-100 по скору
    idx_alert = np.argsort(proba_test)[-100:][::-1]
    print(f"алертов >0.5 всего {(proba_test>0.5).sum()}, берём top-100 по скору")
else:
    # сортируем по убыванию скора и берём 100
    order = np.argsort(proba_test[idx_alert])[::-1][:100]
    idx_alert = idx_alert[order]
    print(f"алертов >0.5: {(proba_test>0.5).sum()}, берём top-100 по скору")
idx_alert = idx_alert[:100]
alert_scores = proba_test[idx_alert]
alert_tx = test_tx[idx_alert]
alert_times = test_times[idx_alert]
print(f"alerts {len(alert_tx)}  score min={alert_scores.min():.3f} med={np.median(alert_scores):.3f} max={alert_scores.max():.3f}")

# feature importances → SHAP-proxy
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=feat_cols).sort_values(ascending=False)
top5_global = feat_imp.head(5)
print("\nTop-5 global importances (SHAP-proxy):")
display(top5_global.to_frame("importance").style.format("{:.5f}"))

# helper: tier
def tier_of(s: float) -> str:
    return "high" if s > 0.8 else "medium" if s > 0.5 else "low"

# сбор Evidence JSON
evidence_list = []
evidence_hashes = []  # SHA256 hex для Merkle
for rank, (idx, txid, score, tstep) in enumerate(zip(idx_alert, alert_tx, alert_scores, alert_times)):
    # anchor: первый сосед из edgelist с фичами, иначе ближайший по Euclidean к train illicit centroid (fallback)
    cands = [n for n in nbr_map.get(int(txid), []) if n in feat_by_id]
    if cands:
        anchor_id = int(cands[0])  # детерминированно первый
        # Euclidean distance
        d = float(np.linalg.norm(feat_by_id[int(txid)] - feat_by_id[anchor_id]))
        anchor_source = "edgelist"
    else:
        # fallback: nearest train illicit by Euclidean (дорого — берём centroid distance прокси)
        # для скорости: distance до train centroid
        centroid = X_train[y_train==1].mean(axis=0)
        anchor_id = int(train_df[y_train==1].iloc[0]["txId"])
        d = float(np.linalg.norm(feat_by_id[int(txid)] - feat_by_id[anchor_id]))
        anchor_source = "centroid_fallback"
    # causal_path effect — proxy: score * (1 - normalized distance)
    effect = float(np.clip(score * (1 - d / (d + 10)), 0, 1))
    # shap_proxy — топ-5 глобальных, взвешенных на score
    shap_vals = {k: float(v * score) for k, v in top5_global.items()}
    ev = {
        "alert_id": f"ALT-{int(txid):08d}-{rank:03d}",
        "risk_score": round(float(score), 4),
        "tier": tier_of(float(score)),
        "anchors": {"distance": round(d, 4), "source": anchor_source, "txId": int(txid), "anchor_txId": anchor_id},
        "causal_path": {"edge": f"{int(txid)}->{anchor_id}", "effect": round(effect, 4)},
        "shap_proxy": {k: round(v, 5) for k, v in shap_vals.items()},
        "provenance": {
            "model_version": "rf100-temporal-v1",
            "date": "2026-09-16",
            "source": "elliptic_raw",
            "train_range": "1..30",
            "test_range": "41..49",
            "time_step": int(tstep),
            "feature_dim": len(feat_cols),
        },
    }
    evidence_list.append(ev)
    # hash для Merkle (canonical JSON)
    canon = json.dumps(ev, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    h = hashlib.sha256(canon.encode()).hexdigest()
    evidence_hashes.append(h)

print(f"\nEvidence: {len(evidence_list)} алертов, пример anchors distance stats: min={min(e['anchors']['distance'] for e in evidence_list):.2f} max={max(e['anchors']['distance'] for e in evidence_list):.2f}")
# гистограмма скоров
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.histplot(alert_scores, bins=20, kde=True, color="steelblue", ax=ax)
ax.set_title("Распределение risk_score 100 алертов (RF, test 41..49, score>0.5)")
ax.set_xlabel("risk_score")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()


## 2. Пример Evidence JSON и валидация схемы

Показываем один алерт (pretty-print) и проверяем обязательные поля. В проде — `jsonschema`/`pydantic`; здесь — lightweight proxy-валидатор без зависимости: проверяем `required` и типы. Нарушение — `raise ValidationError`.


In [ ]:
import textwrap

# pretty print первого алерта
ev0 = evidence_list[0]
print(f"Alert #0  {ev0['alert_id']}  score={ev0['risk_score']}  tier={ev0['tier']}")
print(json.dumps(ev0, indent=2, ensure_ascii=False, sort_keys=False))

# jsonschema proxy — проверка обязательных полей
REQUIRED_SCHEMA = {
    "required": ["alert_id", "risk_score", "tier", "anchors", "causal_path", "shap_proxy", "provenance"],
    "anchors_required": ["distance", "source"],
    "causal_required": ["edge", "effect"],
    "provenance_required": ["model_version", "date", "source"],
}

def validate_evidence(ev: dict) -> tuple[bool, str]:
    for field in REQUIRED_SCHEMA["required"]:
        if field not in ev:
            return False, f"missing field: {field}"
    for sub in REQUIRED_SCHEMA["anchors_required"]:
        if sub not in ev["anchors"]:
            return False, f"anchors missing: {sub}"
    for sub in REQUIRED_SCHEMA["causal_required"]:
        if sub not in ev["causal_path"]:
            return False, f"causal_path missing: {sub}"
    for sub in REQUIRED_SCHEMA["provenance_required"]:
        if sub not in ev["provenance"]:
            return False, f"provenance missing: {sub}"
    if not (0 <= ev["risk_score"] <= 1):
        return False, "risk_score out of [0,1]"
    if ev["tier"] not in ("low", "medium", "high"):
        return False, "tier invalid"
    return True, "ok"

ok, msg = validate_evidence(ev0)
print(f"\nВалидация #0: {ok} ({msg})")

# проверим все 100
results = [validate_evidence(e) for e in evidence_list]
print(f"Валидно: {sum(1 for ok,_ in results if ok)}/{len(results)}  ({sum(1 for ok,_ in results if ok)/len(results):.0%})")
if all(ok for ok,_ in results):
    print("100% provenance — все алерты имеют требуемые поля.")

# demo failed validation (удалим поле)
bad = dict(ev0)
bad.pop("anchors")
ok2, msg2 = validate_evidence(bad)
print(f"\nBroken example (без anchors): valid={ok2}  reason='{msg2}' — валидатор ловит.")


## 3. Подпись — HMAC-SHA256 как proxy Ed25519

Для каждого Evidence считаем `h = SHA256(canonical_json)` (canonical = `sort_keys=True`, `separators=(',',':')`), подписываем `sig = HMAC-SHA256(key, h)`. HMAC даёт 32 байта; для mock Ed25519 (64 байта) конкатенируем `sig || HMAC(key, sig)` → 64 байта, base64 ~88 символов. Верификация — пересчёт HMAC и `compare_digest`.

Демо: меняем 1 символ в JSON (`risk_score 0.9→0.8`) — подпись не проходит. В проде ключ — `Ed25519` приватный (HSM), здесь `DEMO_KEY`.


In [ ]:
# helpers: 64B mock sig via double HMAC

def canonical(ev: dict) -> str:
    return json.dumps(ev, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def evidence_hash(ev: dict) -> str:
    return hashlib.sha256(canonical(ev).encode()).hexdigest()


def sign_evidence(ev: dict, key: bytes = DEMO_KEY) -> dict:
    h_hex = evidence_hash(ev)
    # HMAC-SHA256 of hex hash (as bytes)
    sig1 = hmac.new(key, h_hex.encode(), hashlib.sha256).digest()  # 32B
    sig2 = hmac.new(key, sig1, hashlib.sha256).digest()  # 32B
    sig64 = sig1 + sig2  # 64B mock Ed25519
    return {
        "h": h_hex,
        "sig": sig64,
        "sig_b64": base64.b64encode(sig64).decode(),
        "sig_hex": sig64.hex(),  # 128 hex chars = 64B
    }


def verify_evidence(ev: dict, sig64: bytes, key: bytes = DEMO_KEY) -> bool:
    h_hex = evidence_hash(ev)
    sig1 = hmac.new(key, h_hex.encode(), hashlib.sha256).digest()
    sig2 = hmac.new(key, sig1, hashlib.sha256).digest()
    expected = sig1 + sig2
    return hmac.compare_digest(expected, sig64)


# подпишем 100 алертов
signed = []
for ev in evidence_list:
    s = sign_evidence(ev)
    signed.append(s)

print(f"Пример #0 h={signed[0]['h'][:16]}...  sig_hex len={len(signed[0]['sig_hex'])} (128 hex = 64B)  sig_b64 len={len(signed[0]['sig_b64'])}")
print(f"sig_b64 #0: {signed[0]['sig_b64'][:60]}...")
# верификация всех
verified = [verify_evidence(ev, s["sig"]) for ev, s in zip(evidence_list, signed)]
print(f"\nВерифицировано: {sum(verified)}/{len(verified)} ({np.mean(verified):.0%})")

# tamper demo — меняем 1 символ (risk_score)
tampered = dict(ev0)
tampered["risk_score"] = round(float(tampered["risk_score"]) - 0.01, 4) if tampered["risk_score"] > 0.05 else round(float(tampered["risk_score"]) + 0.01, 4)
print(f"\nОригинал risk_score={ev0['risk_score']}  tampered={tampered['risk_score']}")
print(f"verify(original_sig, original_ev) = {verify_evidence(ev0, signed[0]['sig'])}")
print(f"verify(original_sig, tampered_ev) = {verify_evidence(tampered, signed[0]['sig'])}  ← 1 символ изменён → fail")
print(f"hash original {evidence_hash(ev0)[:16]}...  tampered {evidence_hash(tampered)[:16]}...  отличаются")

# график: верификация
fig, ax = plt.subplots(figsize=(5, 3.2))
sns.barplot(x=["verified", "tampered_fail"], y=[sum(verified), 1], hue=["verified","tampered_fail"], palette=["seagreen","tomato"], legend=False, ax=ax)
ax.set_title("Верификация подписи (HMAC-SHA256 proxy Ed25519)")
ax.set_ylabel("count")
for i, v in enumerate([sum(verified), 1]):
    ax.text(i, v+1, str(v), ha="center", fontsize=11)
plt.tight_layout()
plt.show()


## 4. Merkle-дерево: root и proof для алерта #0

Из 100 `h = SHA256(Evidence)` строим бинарное дерево: pairwise `SHA256(left || right)` (байты), при нечётном — дублируем последний. Root — 32 байта. Для алерта #0 генерируем proof — `⌈log₂100⌉ = 7` хешей-сиблингов с направлением. Верификация — подъём от листа к root.

Зачем: вместо 100 подписей достаточно заякорить один root (OTS→Bitcoin). Проверяющий получает Evidence + proof (7 хешей) и убеждается, что алерт входит в заякоренный набор.


In [ ]:
# Merkle tree (pairwise SHA256 on bytes)

def build_merkle(leaves_hex: list[str]):
    """Build tree, return (levels, root). levels[0]=leaves, levels[-1]=[root]."""
    if not leaves_hex:
        raise ValueError("empty leaves")
    levels = [leaves_hex[:]]
    cur = leaves_hex[:]
    while len(cur) > 1:
        nxt = []
        for i in range(0, len(cur), 2):
            left = cur[i]
            right = cur[i + 1] if i + 1 < len(cur) else left  # duplicate last
            parent = hashlib.sha256(bytes.fromhex(left) + bytes.fromhex(right)).hexdigest()
            nxt.append(parent)
        levels.append(nxt)
        cur = nxt
    return levels, levels[-1][0]


def merkle_proof(levels: list[list[str]], leaf_idx: int):
    """Return proof as list of (sibling_hex, sibling_is_left)."""
    proof = []
    idx = leaf_idx
    for lvl in levels[:-1]:  # exclude root level
        is_right = idx % 2 == 1
        sib_idx = idx - 1 if is_right else idx + 1
        if sib_idx < len(lvl):
            sib = lvl[sib_idx]
        else:
            sib = lvl[idx]  # duplicated
        # sibling_is_left = True if sibling is left of current
        sib_is_left = is_right
        proof.append((sib, sib_is_left))
        idx //= 2
    return proof


def verify_proof(leaf_hex: str, proof: list[tuple[str, bool]], root_hex: str) -> bool:
    cur = leaf_hex
    for sib, sib_is_left in proof:
        if sib_is_left:
            cur = hashlib.sha256(bytes.fromhex(sib) + bytes.fromhex(cur)).hexdigest()
        else:
            cur = hashlib.sha256(bytes.fromhex(cur) + bytes.fromhex(sib)).hexdigest()
    return hmac.compare_digest(cur, root_hex)


leaves = evidence_hashes  # 100 hex
levels, root = build_merkle(leaves)
print(f"Leaves {len(leaves)}  levels {len(levels)}  root={root[:16]}...{root[-16:]}")
for i, lvl in enumerate(levels):
    print(f" level {i}: {len(lvl)} nodes  e.g. {lvl[0][:12]}...")

proof0 = merkle_proof(levels, 0)
print(f"\nProof for alert #0: {len(proof0)} hashes (ceil(log2 100)=7)")
for i, (sib, is_left) in enumerate(proof0):
    side = "left " if is_left else "right"
    print(f"  [{i}] {side} {sib[:16]}...")

leaf0 = leaves[0]
ok_proof = verify_proof(leaf0, proof0, root)
print(f"\nverify_proof(leaf #0, proof, root) = {ok_proof}")

# tamper proof — заменим один хеш
bad_proof = proof0.copy()
bad_proof[0] = ("00" * 32, bad_proof[0][1])
print(f"verify_proof с битым sibling = {verify_proof(leaf0, bad_proof, root)}  ← fail")

# tamper leaf
print(f"verify_proof(tampered leaf) = {verify_proof(evidence_hash(tampered), proof0, root)}  ← fail")


In [ ]:
# proof size vs N — log scale, anchor points 100, 1M→20, 1B→30
import math

Ns = np.array([100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000, 1_000_000_000])
proof_sizes = np.ceil(np.log2(Ns)).astype(int)

# для текущего N=100 проверим фактический len(proof0) == 7
print(f"N=100  proof={len(proof0)}  ceil(log2)={math.ceil(math.log2(100))}")
for n, s in zip(Ns, proof_sizes):
    print(f"  N={n:>12,}  proof {s:2d} hashes  (~{s*32} B)")

fig, ax = plt.subplots(figsize=(7.2, 3.8))
sns.lineplot(x=Ns, y=proof_sizes, marker="o", color="indigo", ax=ax)
ax.set_xscale("log")
ax.set_title("Размер Merkle proof vs число алертов N (log scale)")
ax.set_xlabel("N (число алертов, log)")
ax.set_ylabel("proof size (хешей, ceil(log₂N))")
# аннотации 1M и 1B
for n, s in [(1_000_000, 20), (1_000_000_000, 30), (100, len(proof0))]:
    ax.annotate(f"{n/1e6:.0f}M→{s}" if n>=1e6 else f"1B→{s}" if n==1e9 else f"100→{s}",
                xy=(n, s), xytext=(8, 10), textcoords="offset points", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.2", fc="wheat", alpha=0.6),
                arrowprops=dict(arrowstyle="->", color="grey"))
ax.set_ylim(0, 32)
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

# доп: сколько байт экономит Merkle vs хранение всех подписей
print(f"\nБез Merkle: 100 × 64B sig = {100*64} B (+ 100 хешей)")
print(f"С Merkle: root 32B + proof 7×32B=224B на алерт при проверке (log overhead)")


## 5. WORM proxy: append-only лог + anchoring (proxy OpenTimestamps → Bitcoin)

WORM — Write Once Read Many: лог только дописывается, удаление/изменение ломает цепочку `prev_hash`. Каждое событие — `create / view / status_change / SAR` — хешируется и связывается. Root Merkle-дерева якорим в файл `worm_root.txt` — proxy `OpenTimestamps` (в проде — `ots stamp` → Bitcoin `OP_RETURN`).

Audit trail: проверяющий читает `worm_root.txt` (заякоренный root), лог и proof — и убеждается, что история не переписана.


In [ ]:
from pathlib import Path as _Path

# WORM log — append-only list с hash chaining
worm_log: list[dict] = []
prev_hash = "00" * 32  # genesis

def worm_append(event_type: str, alert_id: str, actor: str, details: dict, prev: str = None) -> dict:
    global prev_hash
    if prev is None:
        prev = prev_hash
    entry = {
        "index": len(worm_log),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event_type": event_type,  # create / view / status_change / SAR
        "alert_id": alert_id,
        "actor": actor,
        "details": details,
        "prev_hash": prev,
    }
    # hash entry без поля entry_hash
    canon_entry = json.dumps(entry, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    entry_hash = hashlib.sha256(canon_entry.encode()).hexdigest()
    entry["entry_hash"] = entry_hash
    worm_log.append(entry)
    prev_hash = entry_hash
    return entry

# симулируем audit trail для первых 5 алертов
actors = ["model@spillety", "analyst@spillety", "compliance@spillety", "sar@spillety"]
for i in range(min(5, len(evidence_list))):
    aid = evidence_list[i]["alert_id"]
    worm_append("create", aid, actors[0], {"risk_score": evidence_list[i]["risk_score"], "tier": evidence_list[i]["tier"]})
    worm_append("view", aid, actors[1], {"action": "reviewed anchors", "distance": evidence_list[i]["anchors"]["distance"]})
    worm_append("status_change", aid, actors[2], {"from": "open", "to": "escalated" if evidence_list[i]["tier"]=="high" else "pending"})
# SAR для #0
worm_append("SAR", evidence_list[0]["alert_id"], actors[3], {"case_id": "SAR-2026-0001", "root_provenance": "rf100-temporal-v1"})
# добавим ещё пачку create для остальных алертов (без view) — чтобы лог 100+ записей
for i in range(5, len(evidence_list)):
    aid = evidence_list[i]["alert_id"]
    worm_append("create", aid, actors[0], {"risk_score": evidence_list[i]["risk_score"]})

print(f"WORM log: {len(worm_log)} записей, типы: {pd.Series([e['event_type'] for e in worm_log]).value_counts().to_dict()}")
print(f"Chain head entry_hash={worm_log[-1]['entry_hash'][:16]}...  prev links ok={all(worm_log[i]['prev_hash']==worm_log[i-1]['entry_hash'] for i in range(1,len(worm_log)))}")

# root anchoring mock (OTS → Bitcoin proxy) — запись в файл
# ищем куда писать: рядом с ноутбуком или в /tmp
anchor_candidates = [_Path("../../worm_root.txt"), _Path("worm_root.txt"), _Path("/tmp/worm_root.txt"), DATA_ROOT.parent / "worm_root.txt"]
anchor_path = None
for p in anchor_candidates:
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        anchor_path = p
        break
    except Exception:
        continue
if anchor_path is None:
    anchor_path = _Path("/tmp/worm_root.txt")
    anchor_path.parent.mkdir(parents=True, exist_ok=True)

anchor_payload = {
    "merkle_root": root,
    "root_b64": base64.b64encode(bytes.fromhex(root)).decode(),
    "n_alerts": len(evidence_list),
    "worm_head": worm_log[-1]["entry_hash"],
    "anchored_at": datetime.now(timezone.utc).isoformat(),
    "proxy": "OpenTimestamps -> Bitcoin (mock, OTS stamp of root)",
}
anchor_path.write_text(json.dumps(anchor_payload, indent=2, ensure_ascii=False))
print(f"\nAnchored root → {anchor_path.resolve()}")
print(anchor_path.read_text()[:600])

# проверим что root в файле совпадает с вычисленным
reloaded = json.loads(anchor_path.read_text())
print(f"\nAnchor root match: {reloaded['merkle_root']==root}  (WORM head {reloaded['worm_head'][:12]}...)")

# audit trail table — последние 8 событий
audit_df = pd.DataFrame(worm_log)[["index","timestamp","event_type","alert_id","actor","prev_hash","entry_hash"]]
# укоротим хеши для таблицы
audit_df["prev"] = audit_df["prev_hash"].str[:10] + "..."
audit_df["hash"] = audit_df["entry_hash"].str[:10] + "..."
display(audit_df.tail(8).style.set_caption("Audit trail — последние 8 записей (append-only)"))

# график событий по типам
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
sns.countplot(data=pd.DataFrame(worm_log), x="event_type", order=["create","view","status_change","SAR"], hue="event_type", palette="colorblind", legend=False, ax=axes[0])
axes[0].set_title("WORM: события по типам")
axes[0].set_ylabel("count")
# timeline — index vs event_type (strip)
sns.stripplot(data=pd.DataFrame(worm_log), x="index", y="event_type", hue="event_type", palette="colorblind", legend=False, ax=axes[1], size=4, alpha=0.7)
axes[1].set_title("WORM timeline (index → event)")
plt.tight_layout()
plt.show()

print("\nWORM-инвариант: удаление любой записи ломает prev_hash цепочку; изменение Evidence ломает Merkle proof → root в worm_root.txt не совпадёт. Это append-only + anchoring, аналог OTS→Bitcoin.")


## 6. Метрики: время верификации, размер proof, доля верифицированных

Мерим микробенчмарком (100 верификаций подписи + 100 Merkle-proof), считаем `proof size`, `verification time` и `share verified`. Все 100 алертов должны быть `verified=100%` — 100% provenance.


In [ ]:
# микробенчмарк: 100 подписей + 100 Merkle proofs
n_bench = 100

# подпись
t0 = time.perf_counter()
for ev, s in zip(evidence_list, signed):
    verify_evidence(ev, s["sig"])
t_sig = time.perf_counter() - t0

# Merkle proof (для каждого алерта свой proof — бенчмарким #0 повторно n раз, т.к. proof детерминирован)
# для честности сгенерим proof для всех 100
all_proofs = [merkle_proof(levels, i) for i in range(len(leaves))]
t1 = time.perf_counter()
for i, leaf in enumerate(leaves):
    verify_proof(leaf, all_proofs[i], root)
t_merkle = time.perf_counter() - t1

per_sig_us = t_sig / n_bench * 1e6
per_merkle_us = t_merkle / n_bench * 1e6
total_us = per_sig_us + per_merkle_us

print(f"HMAC-SHA256 verify: {n_bench} × {per_sig_us:.1f} µs  (total {t_sig*1e3:.2f} ms)")
print(f"Merkle proof verify (7 hashes): {n_bench} × {per_merkle_us:.1f} µs  (total {t_merkle*1e3:.2f} ms)")
print(f"Combined per-alert: {total_us:.1f} µs  (~{1e6/total_us:.0f} алертов/сек на одном ядре)")

# proof size
proof_bytes = len(proof0) * 32  # 7 * 32
sig_bytes = 64
print(f"\nProof size #0: {len(proof0)} hashes ×32B = {proof_bytes} B  (+ leaf 32B)")
print(f"Sig size: {sig_bytes} B (mock Ed25519)")
print(f"Anchored: root 32B в worm_root.txt (vs {len(evidence_list)*sig_bytes} B без Merkle)")

# доля верифицированных
share_verified = np.mean(verified)
share_merkle = np.mean([verify_proof(l, p, root) for l, p in zip(leaves, all_proofs)])
print(f"\nДоля верифицированных подписей: {share_verified:.0%}  ({sum(verified)}/{len(verified)})")
print(f"Доля верифицированных Merkle:   {share_merkle:.0%}  ({int(share_merkle*len(leaves))}/{len(leaves)})")

# сводная таблица метрик
metrics = pd.DataFrame([
    {"metric": "n_alerts", "value": len(evidence_list), "unit": "шт"},
    {"metric": "proof_size", "value": len(proof0), "unit": "хешей (7×32B)"},
    {"metric": "sig_size", "value": sig_bytes, "unit": "B (mock Ed25519)"},
    {"metric": "root_size", "value": 32, "unit": "B"},
    {"metric": "verify_sig", "value": round(per_sig_us, 1), "unit": "µs/алерт"},
    {"metric": "verify_merkle", "value": round(per_merkle_us, 1), "unit": "µs/алерт"},
    {"metric": "share_verified", "value": f"{share_verified:.0%}", "unit": ""},
    {"metric": "worm_entries", "value": len(worm_log), "unit": "записей"},
])
display(metrics.style.set_caption("Метрики аудита (proxy, без внешних зависимостей)").hide(axis="index"))

# графики метрик
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
# время верификации
sns.barplot(x=["HMAC verify","Merkle verify","combined"], y=[per_sig_us, per_merkle_us, total_us], hue=["HMAC verify","Merkle verify","combined"], palette=["steelblue","darkorange","seagreen"], legend=False, ax=axes[0])
axes[0].set_title("Время верификации (µs/алерт, 100 итераций)")
axes[0].set_ylabel("µs")
for i, v in enumerate([per_sig_us, per_merkle_us, total_us]):
    axes[0].text(i, v+max(total_us*0.02, 2), f"{v:.1f}", ha="center", fontsize=9)
# proof size scaling (снова, но как метрика)
sns.barplot(x=[f"N={len(leaves)}"], y=[len(proof0)], color="indigo", ax=axes[1])
axes[1].axhline(20, color="grey", linestyle="--", label="1M→20")
axes[1].axhline(30, color="grey", linestyle=":", label="1B→30")
axes[1].set_title("Proof size для N=100 vs шкала (1M→20, 1B→30)")
axes[1].set_ylabel("хешей")
axes[1].legend(fontsize=8)
axes[1].text(0, len(proof0)+0.15, str(len(proof0)), ha="center", fontsize=11, weight="bold")
plt.tight_layout()
plt.show()


## Выводы

- **100% provenance.** Все 100 алертов имеют Evidence JSON с `provenance {model_version, date, source, train/Test}`, валидация схемы — `100%` (`required` поля). Без provenance алерт — мнение; с ним — артефакт для SAR/аудита.
- **Подпись proxy Ed25519.** `HMAC-SHA256 → 64B mock` подписывает `SHA256(canonical_json)`; верификация `compare_digest`, tamper 1 символа → `fail`. В проде — замена на `Ed25519` (HSM, pubkey в реестре) без смены интерфейса; ключ `DEMO_KEY` — только для ноутбука.
- **Merkle + proof.** 100 хешей → root `32B` (pairwise `SHA256`), proof для #0 — `7` хешей (`⌈log₂100⌉`), верификация `verify_proof(leaf, proof, root)`. График `proof size vs N` — `log₂`: `1M→20`, `1B→30` — константно малое доказательство независимо от объёма.
- **WORM + anchoring.** Append-only лог (`prev_hash` цепочка) — удаление ломает цепочку; root заякорен в `worm_root.txt` как proxy `OpenTimestamps → Bitcoin` (`OP_RETURN`). Audit trail: `create / view / status_change / SAR` — все события воспроизводимы проверяющим по одному root.
- **Метрики.** Верификация подписи ~десятки µs/алерт, Merkle-proof ~единицы µs (7 хешей), combined <0.1 ms → >10k алертов/сек на ядре; proof `7×32B=224B`, sig `64B`, anchored `32B` (vs `6.4KB` без Merkle для 100). `share_verified = 100%`.
- **Ограничения и ponytail.** Это proxy: `HMAC` требует секрета, `Ed25519` — публичной верификации; `worm_root.txt` — локальный файл, прод — `ots stamp` → Bitcoin. Следующий шаг Spillety — `Ed25519+Merkle+OTS`: нативный `ed25519`, batch-OTS якорение раз в час, WORM в `S3 Object Lock`/`QLDB`, SAR-пакет — Evidence+sig+proof+OTS-receipt.


## OTS async client (S9)

> `ots_anchor_async` — submit к календарю OpenTimestamps, `ots_verify` — верификация proof. Polling/подтверждение — вызывающая сторона.

In [ ]:
# OTS async submit + verify
import sys
sys.path.insert(0, "../../..")
from spillety.evidence.merkle import ots_anchor_async, ots_verify, merkle_root

# Create test data
leaves = [b"alert_1", b"alert_2", b"alert_3"]
root = merkle_root(leaves)
print(f"Merkle root: {root.hex()}")

# Submit (async, no polling)
result = ots_anchor_async(root.hex())
print(f"Submit result: {result}")

# Verify with fixture proof
fixture_proof = root + b"_ots_fixture"
verified = ots_verify(root.hex(), fixture_proof)
print(f"Verify result: {verified}")

## GigaChat draft-only (S8)

> `build_alert_context` — детерминированный контекст из evidence+tx. `explain_alert` — verify-first → POST → post-validate → draft narrative. Ключ только из env.

In [ ]:
# GigaChat context building (deterministic)
from spillety.evidence.gigachat import build_alert_context

evidence = {
    'score': 0.87,
    'tier': 'tier1',
    'shap_values': {'distance_to_nearest_OFAC': 0.45, 'causal_filter_pass_rate': 0.32},
    'e_value': 2.1,
    'gamma': 1.8,
    'causal_passed': True,
    'anchors': [
        {'wallet': '1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa', 'source': 'OFAC SDN', 'distance': 0.12},
    ],
    'causal_path': [{'edge': 'Wallet -> Mixer', 'effect': 0.78, 'gamma': 1.5}],
    'provenance': {'model_version': 'elliptic_v3', 'encoder_version': 'PCA32', 'calibrator': 'isotonic', 'tau': 0.08}
}
tx = {'hash': 'a1b2...', 'chain': 'BTC', 'sender': '1A1z...', 'receiver': '3J98...', 'amount': '0.5 BTC'}

ctx = build_alert_context(evidence, tx)
print(ctx)

## Evidence bridge: rich context + canonical subset = anchors

> `build_evidence` теперь принимает anchors/causal_path/provenance. Merkle/OTS подписывает ТОЛЬКО anchors (доказуемое утверждение).

In [ ]:
# Rich evidence + canonical subset test
from spillety.evidence.evidence import build_evidence, evidence_hash, verify_evidence

# Poor call (backward compatible)
poor = build_evidence(score=0.87, tier='tier1', shap_values={'a': 0.5}, e_value=2.1, gamma=1.8, causal_passed=True)
print(f"Poor evidence keys: {list(poor.keys())}")

# Rich call
rich = build_evidence(
    score=0.87, tier='tier1',
    shap_values={'distance_to_nearest_OFAC': 0.45},
    e_value=2.1, gamma=1.8, causal_passed=True,
    anchors=[{'wallet': '1A1z...', 'source': 'OFAC SDN', 'distance': 0.12}],
    causal_path=[{'edge': 'Wallet->Mixer', 'effect': 0.78, 'gamma': 1.5}],
    provenance={'model_version': 'v3', 'encoder': 'PCA32'}
)
print(f"Rich evidence keys: {list(rich.keys())}")

# Hash test: anchor tamper changes hash, provenance change does not
h1 = evidence_hash(rich)
rich_tampered = rich.copy()
rich_tampered['anchors'][0]['distance'] = 0.99
h2 = evidence_hash(rich_tampered)
print(f"Anchor tamper changes hash: {h1 != h2}")

rich_prov = rich.copy()
rich_prov['provenance']['model_version'] = 'v4'
h3 = evidence_hash(rich_prov)
print(f"Provenance change changes hash: {h1 != h3}")  # Should be False (provenance not in canonical)